# Detection of Spatial Signatures in the Canton of Zürich

This notebook contains complete code required for a detection of spatial signatures in the Canton of Zürich, Switzerland, limited by its administrative (cantonal) boundary.

It is adapted from `singapore.ipynb` in this repository and follows the same four-stage pipeline (form &rarr; function &rarr; clustering &rarr; figures). A few things differ from the Singapore notebook because Zurich is not Singapore:

- **Geometry (form)**: buildings, streets, water and railway are still retrieved from OpenStreetMap via `osmnx`, exactly as in the Singapore notebook, just for `place = "Kanton Zürich, Switzerland"` instead of `"Singapore"`. Since the canton is a landlocked area of ~1'729 km² (vs. Singapore's ~720 km² island), the enclosure `limit` is the cantonal administrative boundary itself (fetched via `ox.geocode_to_gdf`), not a coastline polygonized from water bodies.
- **Function**: Singapore's function layer relies on `data.gov.sg`, a national open-data portal that has no direct Zurich equivalent covering the same themes. Every function layer here (restaurants, supermarkets, land use, parks, historical monuments) is instead sourced from OpenStreetMap, so the whole notebook only depends on one external data provider.
- **Population**: from WorldPop gridded population estimates for Switzerland (`CHE`), same product and URL pattern as Singapore's (`SGP`).
- **NDVI and night lights** are left as `TODO` stubs. The Singapore notebook's NDVI step depends on a local tile index (`data_processing/imagery/GHS-composite-S2.geojson`) that isn't tracked in this repository, and its night-lights step requires a manual, login-gated download from eogdata.mines.edu. Neither carries over automatically; both sections describe what would need to be plugged in.

## Environment

This notebook was developed against a **dedicated virtual environment** (`.venv` at the repository root, Python 3.9), not the system/global Python install. Package versions are pinned deliberately:

- `geopandas==0.13.2`, `osmnx==1.9.4`, `momepy==0.6.0` &mdash; momepy 0.6.0 is the last release that supports Python 3.9 (momepy>=0.7 requires Python>=3.10), and it relies on `GeoDataFrame.sindex.query_bulk`, which was removed in geopandas 1.0. Newer osmnx (2.x) requires geopandas>=1.0.1 and renamed `geometries_from_place`/`get_undirected`, so osmnx is pinned to the last 1.x release too.
- `numpy<2` &mdash; the `pandana` package (used for network-based POI accessibility) ships a compiled extension that isn't compatible with NumPy 2.x yet.

Full pinned list: `code/requirements_zurich.txt`. To reproduce: `python -m venv .venv`, then `.venv\Scripts\pip install -r code/requirements_zurich.txt`, and select the `.venv` interpreter (registered Jupyter kernel: **"Python (fleischmann_signatures)"**).

## Contents

- 1. Analysis of form
    - 1.1. Data retrieval
        - 1.1.1. Building footprints
        - 1.1.2. Street network
        - 1.1.3. Spatial barriers (water, railway, cantonal boundary)
    - 1.2. Data pre-processing
        - 1.2.1. Footprints checks and cleaning
        - 1.2.2. Barriers checks and cleaning
    - 1.3. Generation of geographies
        - 1.3.1. Enclosures
        - 1.3.2. Enclosed tessellation
    - 1.4. Morphometric analysis
        - 1.4.1. Primary morphometric characters
        - 1.4.2. Contextualisation
- 2. Analysis of function (all from OpenStreetMap)
    - 2.1. Eating establishments
    - 2.2. Supermarkets
    - 2.3. Land use
    - 2.4. Parks
    - 2.5. Historical monuments
    - 2.6. Population (WorldPop)
    - 2.7. NDVI (TODO)
    - 2.8. Night lights (TODO)
- 3. Cluster analysis
    - 3.1. Form
    - 3.2. Function
    - 3.3. Spatial Signatures
- 4. Figures
- 5. Cluster summary

All packages and custom modules are imported below.


In [ ]:
import os
import pickle
import warnings

import geopandas as gpd
import dask_geopandas
import libpysal
import momepy
import numpy as np
import osmnx as ox
import pandana
import pandas as pd
import rasterstats
import rioxarray
import scipy
import utils
import utils.tobler
import xarray
from clustergram import Clustergram
from dask.distributed import Client, LocalCluster
from download import download
from shapely.geometry import box, mapping
from shapely.ops import polygonize
from sklearn import preprocessing
from sklearn.cluster import MiniBatchKMeans
from tqdm import tqdm
from utils.tobler import area_max
from utils.dask_geopandas import dask_dissolve

import urbangrammar_graphics as ugg
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
import contextily

Most of our data will be stored as geospatial parquet files. As in the Singapore notebook, we filter the associated (harmless) warning.

In [ ]:
warnings.filterwarnings('ignore', message='.*initial implementation of Parquet.*')


First we specify a path to a folder for data storage (a local folder within the repository, since the original `urbangrammar_samba` network share isn't available on this machine) and the OSM place name for the canton.

In [ ]:
folder = "../data/zurich/"
os.makedirs(folder, exist_ok=True)

place = "Kanton Zürich, Switzerland"

# Canton Zürich is ~1'729 km² (roughly 2.4x the area of Singapore), so Overpass
# queries below can take a while and are more likely to hit default timeouts.
ox.settings.timeout = 1800
ox.settings.memory = None


## 1. Analysis of form

First step towards spatial signatures is an analysis of urban form to capture its patterns.

### 1.1. Data retrieval

All data used in this section are obtained from OpenStreetMap.

#### 1.1.1 Building footprints

Building footprints are obtained from OSM using OSMnx, exactly as in the Singapore notebook.

If this single Overpass query times out or is rejected for being too large (a real risk over a ~1'700 km² canton), a more robust fallback is to download a regional `.osm.pbf` extract (e.g. from Geofabrik) and clip/parse it locally with `pyrosm` or `osmium` instead of querying the live Overpass API. That path isn't implemented here to keep parity with the Singapore notebook's approach.

In [ ]:
%%time
path = folder + 'footprints_osm.pq'
if not os.path.exists(path):
    footprints = ox.geometries_from_place(place, tags={'building': True})
    footprints_proj = ox.projection.project_gdf(footprints)
    cols = [c for c in ['building', 'building:levels', 'geometry'] if c in footprints_proj.columns]
    footprints_proj[cols].to_parquet(path)

#### 1.1.2 Street network

Street network is obtained from OSM using OSMnx, and the same custom tag filter used for Singapore. We get a graph, project it, turn it into an undirected one and export it to a GeoDataFrame. Then we make sure `highway` is a string column so it can be saved to file.

Note: `network_type='all'` is used instead of the deprecated `'all_private'` alias (renamed in osmnx, though still accepted in 1.9.x with a warning).

In [ ]:
%%time
path = folder + 'roads_osm.pq'
if not os.path.exists(path):
    type_filter = '["highway"~"living_street|motorway|motorway_link|pedestrian|primary|primary_link|residential|secondary|secondary_link|service|tertiary|tertiary_link|trunk|trunk_link"]'

    streets_graph = ox.graph_from_place(place, network_type='all', custom_filter=type_filter, retain_all=True)
    streets_graph = ox.projection.project_graph(streets_graph)
    gdf = ox.graph_to_gdfs(ox.get_undirected(streets_graph), nodes=False, edges=True, node_geometry=False, fill_edge_geometry=True)
    gdf.highway = gdf.highway.astype(str)
    gdf[['highway', 'geometry']].to_parquet(path)

#### 1.1.3. Spatial barriers

Identification of enclosures should optimally contain a variety of barriers, not only the street network. Therefore, we also download water bodies and railway geometry from OSM, and the cantonal administrative boundary itself (used below as the outer `limit`, since &mdash; unlike Singapore &mdash; Zürich is landlocked and has no coastline to polygonize into an outline).

##### Water

In [ ]:
%%time
path = folder + 'water_osm.pq'
if not os.path.exists(path):
    tags = {'natural': ['water', 'coastline', 'spring']}
    gdf = ox.geometries_from_place(place, tags)
    gdf[['natural', 'geometry']].to_parquet(path)

##### Railway

Then we get railway and filter out the proper geometry type (we want LineStrings representing railway tracks) and remove tunnels (those are not spatial barriers).

In [ ]:
path = folder + 'railway_osm.pq'
if not os.path.exists(path):
    tags = {'railway': True}
    gdf = ox.geometries_from_place(place, tags)
    gdf = gdf[gdf.geom_type == 'LineString']
    gdf = gdf[gdf.get('tunnel') != 'yes'] if 'tunnel' in gdf.columns else gdf
    gdf[['railway', 'geometry']].to_parquet(path)

##### Cantonal boundary

Used as the outer `limit` for enclosures below (Singapore's notebook instead polygonized water/coastline into an island outline &mdash; not applicable to a landlocked canton).

In [ ]:
path = folder + 'admin_osm.pq'
if not os.path.exists(path):
    admin = ox.geocode_to_gdf(place)
    admin.to_parquet(path)

### 1.2. Data pre-processing

Before we can analyse form using the data we just downloaded, we need to ensure that they are in a proper shape.

#### 1.2.1. Footprints checks and cleaning

Let's start with building footprints.

In [ ]:
buildings = gpd.read_parquet(folder + 'footprints_osm.pq')


OSM data tend to be messy, so we need to check validity and geometry types.

In [ ]:
buildings.geom_type.value_counts()


We explode geometries and keep only polygons.

In [ ]:
buildings = buildings.reset_index(drop=True).explode(index_parts=False).reset_index(drop=True)
buildings = buildings[buildings.geom_type == 'Polygon']


We can then use `momepy.CheckTessellationInput()` to check for potential issues which may arise during enclosed tessellation.

In [ ]:
%time check = momepy.CheckTessellationInput(buildings)


As in the Singapore notebook, we drop collapsed and overlapping geometries rather than manually fixing them.

In [ ]:
buildings = buildings.drop(check.collapse.index.union(check.overlap.index))


In [ ]:
buildings.shape


In [ ]:
buildings.is_valid.all()


We assign a unique ID to each remaining building and save them to a file.

In [ ]:
buildings["uID"] = range(len(buildings))
%time buildings.to_parquet(folder + "buildings.pq")


#### 1.2.2. Barriers checks and cleaning

Let's now check our barriers and close potential unwanted gaps to generate clean enclosures.

In [ ]:
roads = gpd.read_parquet(folder + "roads_osm.pq").to_crs(buildings.crs)


In [ ]:
roads.geom_type.unique()


In [ ]:
railway = gpd.read_parquet(folder + "railway_osm.pq").to_crs(buildings.crs)


In [ ]:
railway.geom_type.unique()


Roads seem fine, similarly to railway. The only step we do is extension of railway lines to snap to roads.

In [ ]:
extended_railway = momepy.extend_lines(railway, 30, target=roads, extension=.1)


In [ ]:
water = gpd.read_parquet(folder + "water_osm.pq").to_crs(buildings.crs)


In [ ]:
water.geom_type.value_counts()


We create LineString geometry out of polygons, so lakes and rivers act as barriers rather than areas.

In [ ]:
mask = water.geom_type.isin(["MultiPolygon", "Polygon"])


In [ ]:
water.loc[mask, "geometry"] = water[mask].boundary


We read the cantonal boundary and reproject it to match. This is the `limit` used for enclosures below &mdash; the outer extent of the study area &mdash; taking the place of Singapore's coastline-derived island outline.

In [ ]:
admin = gpd.read_parquet(folder + "admin_osm.pq").to_crs(buildings.crs)
limit = admin.geometry.unary_union


### 1.3. Generation of geographies

Now we create the geographies we will be using in the analysis: enclosures and enclosed tessellation.

#### 1.3.1. Enclosures

Enclosures require spatial barriers &mdash; roads, water and railway (using the extended one from above) &mdash; limited by the cantonal boundary.

In [ ]:
%%time
path = folder + "enclosures.pq"
if os.path.exists(path):
    enclosures = gpd.read_parquet(path)
else:
    enclosures = momepy.enclosures(roads, limit=gpd.GeoSeries([limit], crs=buildings.crs), additional_barriers=[extended_railway, water])
    enclosures.to_parquet(path)

In [ ]:
enclosures.is_valid.all()


#### 1.3.2. Enclosed tessellation

With enclosures and buildings, we can now create the enclosed tessellation. To have better control of the dask cluster, we create it directly (enclosed tessellation is a dask-parallelised function). Adjust `n_workers` to your machine.

In [ ]:
client = Client(LocalCluster(n_workers=8))
client


In [ ]:
%%time
path = folder + "tessellation.pq"
if os.path.exists(path):
    tessellation = gpd.read_parquet(path)
else:
    tess = momepy.Tessellation(buildings, 'uID', enclosures=enclosures)
    tessellation = tess.tessellation
    tessellation['tID'] = range(len(tessellation))
    tessellation.to_parquet(path)

Now we can assign a unique ID to each tessellation cell and save it.

### 1.4. Morphometric analysis

Morphometric analysis first measures primary characters, which are then contextualised to capture the tendencies within the local context of each tessellation cell.

#### 1.4.1. Primary morphometric characters

For the analysis we need four input layers representing urban form: buildings, streets, enclosed tessellation, and enclosures.

In [ ]:
tess = gpd.read_parquet(folder + "tessellation.pq")
blg = gpd.read_parquet(folder + "buildings.pq")
streets = gpd.read_parquet(folder + 'roads_osm.pq')
enclosures = gpd.read_parquet(folder + "enclosures.pq")


In [ ]:
tess.crs == blg.crs == streets.crs == enclosures.crs


In many cases, we will need to relate buildings to tessellation cells. The best way to ensure a proper linkage is to combine both GeoDataFrames into a single one with two named geometry columns. Not every cell has an assigned building.

In [ ]:
tess = tess.rename_geometry("tessellation").merge(blg[['uID', 'geometry']].rename_geometry("buildings"), on='uID', how='left')


In [ ]:
tess


From the combined GeoDataFrame, we can create one with buildings as the active geometry, dropping rows without a building.

In [ ]:
blg = tess.set_geometry('buildings').dropna()


Now we can measure morphometric characters. Refer to the [momepy documentation](https://docs.momepy.org) for details on each one. Each result is assigned as a new column. (This mirrors the Singapore notebook's cell exactly, using `momepy.*` directly &mdash; the custom `utils.momepy` wrappers this repository used in 2021 have since been merged upstream into momepy itself.)

In [ ]:
%time blg['sdbAre'] = momepy.Area(blg).series
%time blg['sdbPer'] = momepy.Perimeter(blg).series
%time blg['sdbCoA'] = momepy.CourtyardArea(blg, 'sdbAre').series

%time blg['ssbCCo'] = momepy.CircularCompactness(blg, 'sdbAre').series
%time blg['ssbCor'] = momepy.Corners(blg).series
%time blg['ssbSqu'] = momepy.Squareness(blg).series
%time blg['ssbERI'] = momepy.EquivalentRectangularIndex(blg, 'sdbAre', 'sdbPer').series
%time blg['ssbElo'] = momepy.Elongation(blg).series
%time cencon = momepy.CentroidCorners(blg)
blg['ssbCCM'] = cencon.mean
blg['ssbCCD'] = cencon.std

%time blg['stbOri'] = momepy.Orientation(blg).series

%time tess['stcOri'] = momepy.Orientation(tess).series

tess = tess.merge(blg[['tID', 'stbOri']], on='tID', how='left')

%time tess['stbCeA'] = (tess['stbOri'] - tess['stcOri']).abs()

%time tess['sdcLAL'] = momepy.LongestAxisLength(tess).series
%time tess['sdcAre'] = momepy.Area(tess).series
%time tess['sscCCo'] = momepy.CircularCompactness(tess, 'sdcAre').series
%time tess['sscERI'] = momepy.EquivalentRectangularIndex(tess, 'sdcAre').series

%time tess['sicCAR'] = tess.buildings.area / tess['sdcAre']
%time blg["mtbSWR"] = momepy.SharedWallsRatio(blg).series


Some characters require spatial weights matrices. We generate Queen contiguity based on the enclosed tessellation.

In [ ]:
%time queen_1 = libpysal.weights.contiguity.Queen.from_dataframe(tess, ids="tID", geom_col='tessellation')
%time tess["mtbAli"] = momepy.Alignment(tess.set_geometry("buildings"), queen_1, "tID", "stbOri").series
%time tess["mtbNDi"] = momepy.NeighborDistance(tess.set_geometry("buildings"), queen_1, "tID").series
%time tess["mtcWNe"] = momepy.Neighbors(tess, queen_1, "tID", weighted=True).series
%time tess["mdcAre"] = momepy.CoveredArea(tess, queen_1, "tID").series
%time blg_q1 = libpysal.weights.contiguity.Queen.from_dataframe(blg, geom_col='buildings', silence_warnings=True)

%time blg["ldbPWL"] = momepy.PerimeterWall(blg, blg_q1).series
%time blg["libNCo"] = momepy.Courtyards(blg, spatial_weights=blg_q1).series
%time queen_3 = momepy.sw_high(k=3, weights=queen_1)
%time tess['ltbIBD'] = momepy.MeanInterbuildingDistance(tess.set_geometry('buildings'), queen_1, 'tID', queen_3).series


We also need to link tessellation cells to streets, based on intersection. This gives us the ratio if the tessellation intersects more than one street segment. From these ratios we get the primary link (the one intersecting the most). Since we need these links on both tessellation and buildings, we merge the result into `blg`.

In [ ]:
# get nid and primary nid here
%time links = momepy.get_network_ratio(tess, streets)
tess[['edgeID_keys', 'edgeID_values']] = links
keys = tess.edgeID_values.apply(lambda a: np.argmax(a))
tess['edgeID_primary'] = [inds[i] for inds, i in zip(tess.edgeID_keys, keys)]
blg = blg.merge(tess[['tID', 'edgeID_primary']], on='tID', how='left')
streets['edgeID_primary'] = range(len(streets))


In [42]:
%time streets["sdsLen"] = momepy.Perimeter(streets).series
%time tess["stcSAl"] = momepy.StreetAlignment(tess, streets, "stcOri", "edgeID_primary").series
%time blg["stbSAl"] = momepy.StreetAlignment(blg, streets, "stbOri", "edgeID_primary").series


CPU times: total: 15.6 ms
Wall time: 15 ms
CPU times: total: 41.3 s
Wall time: 42.5 s
CPU times: total: 40.9 s
Wall time: 42.2 s


In [ ]:
%time profile = momepy.StreetProfile(streets, blg, distance=3)
streets["sdsSPW"] = profile.w
streets["sdsSPO"] = profile.o
streets["sdsSWD"] = profile.wd


In [ ]:
%time streets["sssLin"] = momepy.Linearity(streets).series

In [ ]:
%%time
# Area covered by each edge
vals = {x: [] for x in range(len(streets))}
for i, keys in enumerate(tess.edgeID_keys):
    for k in keys:
        vals[k].append(i)
area_sums = []
for inds in vals.values():
    area_sums.append(tess.sdcAre.iloc[inds].sum())
streets['sdsAre'] = area_sums


In [ ]:
%%time
# Buildings per meter
bpm = []
for inds, l in zip(vals.values(), streets.sdsLen):
    bpm.append(tess.buildings.iloc[inds].notna().sum() / l if len(inds) > 0 else 0)
streets['sisBpM'] = bpm


The street network forms a graph. We can use this property to first define a context of each segment.

In [ ]:
str_q1 = libpysal.weights.contiguity.Queen.from_dataframe(streets, silence_warnings=True)

streets["misRea"] = momepy.Reached(
    streets, tess, "edgeID_primary", "edgeID_primary", spatial_weights=str_q1, mode="count"
).series
streets["mdsAre"] = momepy.Reached(streets, tess, "edgeID_primary", "edgeID_primary", spatial_weights=str_q1,
                               mode="sum").series


Then we create a graph representation and measure connectivity characters. The cell below creates a `networkx.MultiGraph`, measures characters and returns two GeoDataFrames &mdash; original segments and nodes &mdash; accompanied by spatial weights.

In [ ]:
%time graph = momepy.gdf_to_nx(streets)

print("node degree")
graph = momepy.node_degree(graph)

print("subgraph")
graph = momepy.subgraph(
    graph,
    radius=5,
    meshedness=True,
    cds_length=False,
    mode="sum",
    degree="degree",
    length="mm_len",
    mean_node_degree=False,
    proportion={0: True, 3: True, 4: True},
    cyclomatic=False,
    edge_node_ratio=False,
    gamma=False,
    local_closeness=True,
    closeness_weight="mm_len",
)
print("cds length")
graph = momepy.cds_length(graph, radius=3, name="ldsCDL")

print("clustering")
graph = momepy.clustering(graph, name="xcnSCl")

print("mean_node_dist")
graph = momepy.mean_node_dist(graph, name="mtdMDi")

%time nodes, edges, sw = momepy.nx_to_gdf(graph, spatial_weights=True)


Weights and connections allow us to measure characters on a "subgraph" even without the networkX object.

In [ ]:
%time edges_w3 = momepy.sw_high(k=3, gdf=edges)
%time edges["ldsMSL"] = momepy.SegmentsLength(edges, spatial_weights=edges_w3, mean=True).series

%time nodes_w5 = momepy.sw_high(k=5, weights=sw)
%time nodes["lddNDe"] = momepy.NodeDensity(nodes, edges, nodes_w5).series
nodes["linWID"] = momepy.NodeDensity(
    nodes, edges, nodes_w5, weighted=True, node_degree="degree"
).series


Now we also include enclosures in the set.

In [ ]:
enclosures["ldeAre"] = momepy.Area(enclosures).series
enclosures["ldePer"] = momepy.Perimeter(enclosures).series
enclosures["lseCCo"] = momepy.CircularCompactness(enclosures, "ldeAre").series
enclosures["lseERI"] = momepy.EquivalentRectangularIndex(enclosures, "ldeAre", "ldePer").series
enclosures["lseCWA"] = momepy.CompactnessWeightedAxis(enclosures, "ldeAre", "ldePer").series
enclosures["lteOri"] = momepy.Orientation(enclosures).series

blo_q1 = libpysal.weights.contiguity.Queen.from_dataframe(enclosures, ids="eID")

inp, res = enclosures.sindex.query_bulk(enclosures.geometry, predicate='intersects')
indices, counts = np.unique(inp, return_counts=True)
enclosures['neighbors'] = counts - 1
enclosures['lteWNB'] = enclosures['neighbors'] / enclosures['ldePer']


In [ ]:
# Measure weighted cells within enclosure
encl_counts = tess.groupby('eID').count()
merged = enclosures[['eID', 'ldeAre']].merge(encl_counts[['tessellation']], how='left', on='eID')
enclosures['lieWCe'] = merged['tessellation'] / merged['ldeAre']


In [ ]:
tess['ltcWRE'] = momepy.BlocksCount(tess, 'eID', queen_3, 'tID').series


We will need to link data on nodes to tessellation, so we need to get the nearest network-based node ID.

In [ ]:
# get node id
%time links = momepy.get_network_ratio(tess, edges)
tess[['edgeID_keys2', 'edgeID_values2']] = links
%time tess['nodeID'] = momepy.get_node_id(tess, nodes, edges, node_id='nodeID', edge_keys='edgeID_keys2', edge_values='edgeID_values2')


In [ ]:
%%time
nodes["sddAre"] = momepy.Reached(
    nodes, tess, "nodeID", "nodeID", mode="sum", values="sdcAre"
).series


In [ ]:
tess.to_parquet(folder + "tess.pq")
blg.to_parquet(folder + "blg.pq")
nodes.to_parquet(folder + "nodes.pq")
edges.to_parquet(folder + "edges.pq")
enclosures.to_parquet(folder + "enclosures.pq")


Finally, we link all GeoDataFrames together.

In [ ]:
data = tess.merge(
    blg[
        [
            "tID",
            "sdbAre",
            "sdbPer",
            "sdbCoA",
            "ssbCCo",
            "ssbCor",
            "ssbSqu",
            "ssbERI",
            "ssbElo",
            "ssbCCM",
            "ssbCCD",
            "mtbSWR",
            "ldbPWL",
            "stbSAl",
            "libNCo",
        ]
    ],
    on="tID",
    how="left",
)
data = data.merge(
    edges[
        [
            "sdsLen",
            "sdsSPW",
            "sdsSPO",
            "sdsSWD",
            "sssLin",
            "sdsAre",
            "sisBpM",
            "misRea",
            "mdsAre",
            "ldsMSL",
            "edgeID_primary",
        ]
    ],
    on="edgeID_primary",
    how="left",
)
data = data.merge(
    nodes[
        [
            "degree",
            "meshedness",
            "proportion_3",
            "proportion_4",
            "proportion_0",
            "local_closeness",
            "ldsCDL",
            "xcnSCl",
            "mtdMDi",
            "nodeID",
            "lddNDe",
            "linWID",
            "sddAre",
        ]
    ],
    on="nodeID",
    how="left",
)
data = data.merge(
    enclosures[
        [
            "eID",
            "ldeAre",
            "ldePer",
            "lseCCo",
            "lseERI",
            "lseCWA",
            "lteOri",
            "lteWNB",
            "lieWCe",
        ]
    ],
    on="eID",
    how="left",
)


In [ ]:
data.columns


Now we specify columns representing characters and nothing else (geometry, IDs).

In [ ]:
characters = [
    "stcOri",
    "stbOri",
    "stbCeA",
    "sdcLAL",
    "sdcAre",
    "sscCCo",
    "sscERI",
    "sicCAR",
    "mtbAli",
    "mtbNDi",
    "mtcWNe",
    "mdcAre",
    "ltbIBD",
    "stcSAl",
    "ltcWRE",
    "sdbAre",
    "sdbPer",
    "sdbCoA",
    "ssbCCo",
    "ssbCor",
    "ssbSqu",
    "ssbERI",
    "ssbElo",
    "ssbCCM",
    "ssbCCD",
    "mtbSWR",
    "ldbPWL",
    "stbSAl",
    "libNCo",
    "sdsLen",
    "sdsSPW",
    "sdsSPO",
    "sdsSWD",
    "sssLin",
    "sdsAre",
    "sisBpM",
    "misRea",
    "mdsAre",
    "ldsMSL",
    "degree",
    "meshedness",
    "proportion_3",
    "proportion_4",
    "proportion_0",
    "local_closeness",
    "ldsCDL",
    "xcnSCl",
    "mtdMDi",
    "lddNDe",
    "linWID",
    "sddAre",
    "ldeAre",
    "ldePer",
    "lseCCo",
    "lseERI",
    "lseCWA",
    "lteOri",
    "lteWNB",
    "lieWCe",
]


Now we save all data to a parquet file.

In [ ]:
data[characters + ['tID', 'uID', 'tessellation', 'buildings']].to_parquet(folder + "data.pq")


#### 1.4.2. Contextualisation

Cluster analysis needs to be performant on large datasets, so it should not contain spatial constraints (such algorithms do not scale well). At the same time, we are interested in the tendency of characters within an area. Most primary characters are local only, with minimal sense of context. We therefore contextualise each primary character to get the first, second and third quartile of the distribution of values within an inclusive 10th order of contiguity around each tessellation cell, weighted by the inverse distance between cell centroids.

First we load the data (unless it's still in memory from the steps above).

In [ ]:
data = gpd.read_parquet(folder + "data.pq")


We prepare `gdf`, which contains characters and tessellation centroids as geometry, and `W`, a spatial weights matrix representing the 10th order of contiguity (inclusive).

In [ ]:
gdf = gpd.GeoDataFrame(data[characters], geometry=data.tessellation.centroid)
%time W = momepy.sw_high(k=10, weights=libpysal.weights.Queen.from_dataframe(data, geom_col='tessellation'))


Then we loop over the GeoDataFrame and measure the contextualisation of each character. (Note: `.iteritems()` was removed from pandas in 2.0; we use `.items()` instead.)

In [ ]:
convolutions = {}
for c in characters:
    convolutions[c] = []

# measure convolutions
for i, geom in tqdm(gdf.geometry.items(), total=data.shape[0]):
    neighbours = W.neighbors[i]
    vicinity = gdf.iloc[neighbours]
    distance = vicinity.distance(geom)
    distance_decay = (1 / distance).replace(np.inf, 0)  # to filter out duplications

    for c in characters:
        values = vicinity[c].values
        sorter = np.argsort(values)
        values = values[sorter]
        nan_mask = np.isnan(values)
        if nan_mask.all():
            convolutions[c].append(np.array([np.nan] * 3))
        else:
            sample_weight = distance_decay.values[sorter][~nan_mask]
            weighted_quantiles = np.cumsum(sample_weight) - 0.5 * sample_weight
            weighted_quantiles /= np.sum(sample_weight)
            interpolate = np.interp([.25, .5, .75], weighted_quantiles, values[~nan_mask])
            convolutions[c].append(interpolate)


The resulting dictionary can then be exploded into a DataFrame with the tessellation ID as an index.

In [ ]:
%time conv = pd.DataFrame(convolutions, index=data.index)
%time exploded = pd.concat([pd.DataFrame(conv[c].to_list(), columns=[c + '_q1', c + '_q2', c + '_q3']) for c in characters], axis=1)


In [ ]:
exploded.index = data.tID
exploded


And saved to a file, together with the weights matrix.

In [ ]:
exploded.to_parquet(folder + "convolutions.pq")


In [ ]:
scipy.sparse.save_npz(folder + "w10.npz", W.sparse)


## 2. Analysis of function

Next we want to characterise the built environment by its function. Singapore's function layer draws on several `data.gov.sg` datasets that have no single Zurich-wide equivalent, so **every layer below is sourced from OpenStreetMap** instead, keeping the notebook to one external data provider for this whole section.

### 2.1. Eating establishments

We query OSM for common eating-related `amenity` tags. Points (and polygons, whose centroid we take) are linked to tessellation based on accessibility: as in the Singapore notebook, we use `pandana` to measure the number of POIs within a 15 minute walk (represented as 1200 m on the street network).

In [ ]:
tess = gpd.read_parquet(folder + "tess.pq", columns=["tessellation", 'tID'])
edges = gpd.read_parquet(folder + "edges.pq")
nodes = gpd.read_parquet(folder + "nodes.pq")


We build the pandana network from our existing nodes and edges data.

In [ ]:
nodes = nodes.set_index('nodeID')
network = pandana.Network(nodes.geometry.x, nodes.geometry.y,
                          edges['node_start'], edges['node_end'], edges[['mm_len']])


In [ ]:
%%time
eating_tags = {'amenity': ['restaurant', 'cafe', 'fast_food', 'bar', 'pub', 'biergarten', 'food_court']}
eating = ox.geometries_from_place(place, eating_tags)
eating = eating.to_crs(nodes.crs)
eating['geometry'] = eating.geometry.centroid


In [ ]:
%%time
network.set_pois(category='eating',
                 maxdist=1200,
                 maxitems=eating.shape[0],
                 x_col=eating.geometry.x,
                 y_col=eating.geometry.y)


And now we measure the number of points within 1200 m around each node. Notice `count(axis=1)` at the end of the cell.

In [ ]:
%%time
nodes['eating'] = network.nearest_pois(distance=1200,
                               category='eating',
                               num_pois=5000,
                               include_poi_ids=False).replace(1200, pd.NA).count(axis=1)


### 2.2 Supermarkets

Supermarkets are tagged `shop=supermarket` in OSM.

In [ ]:
%%time
supermarket_tags = {'shop': 'supermarket'}
supermarkets = ox.geometries_from_place(place, supermarket_tags)
supermarkets = supermarkets.to_crs(nodes.crs)
supermarkets['geometry'] = supermarkets.geometry.centroid


We can directly link the locations to our existing pandana network as we did before.

In [ ]:
%%time
network.set_pois(category='supermarkets',
                 maxdist=1200,
                 maxitems=supermarkets.shape[0],
                 x_col=supermarkets.geometry.x,
                 y_col=supermarkets.geometry.y)


In [ ]:
%%time
nodes['supermarkets'] = network.nearest_pois(distance=1200,
                               category='supermarkets',
                               num_pois=supermarkets.shape[0],
                               include_poi_ids=False).replace(1200, pd.NA).count(axis=1)


### 2.3 Land Use

Land use is retrieved from OSM's `landuse` tag and simplified to the same four coarse categories Singapore's notebook used (`RESIDENTIAL`, `COMMERCIAL`, `NATURAL`, `OTHER`).

In [ ]:
%%time
land_use = ox.geometries_from_place(place, tags={'landuse': True})
land_use = land_use[land_use.geom_type.isin(['Polygon', 'MultiPolygon'])]


In [ ]:
land_use.landuse.value_counts()


OSM's `landuse` categories are mapped to the four coarse groups. Industrial/retail/commercial land is grouped with `COMMERCIAL` (as Singapore's business-park categories were), farmland/forest/grass/cemeteries with `NATURAL`, and everything else with `OTHER`.

In [ ]:
mapp = {
    "residential": "RESIDENTIAL",
    "commercial": "COMMERCIAL",
    "retail": "COMMERCIAL",
    "industrial": "COMMERCIAL",
    "railway": "OTHER",
    "farmland": "NATURAL",
    "farmyard": "NATURAL",
    "forest": "NATURAL",
    "meadow": "NATURAL",
    "grass": "NATURAL",
    "orchard": "NATURAL",
    "vineyard": "NATURAL",
    "allotments": "NATURAL",
    "greenfield": "NATURAL",
    "cemetery": "NATURAL",
    "recreation_ground": "NATURAL",
    "religious": "OTHER",
    "education": "OTHER",
    "institutional": "OTHER",
    "military": "OTHER",
    "quarry": "OTHER",
    "brownfield": "OTHER",
    "garages": "OTHER",
    "depot": "OTHER",
}
land_use["use"] = land_use.landuse.map(mapp).fillna("OTHER")


Geometry needs validation, then we reproject to match the tessellation.

In [ ]:
land_use.geometry.is_valid.all()


In [ ]:
land_use.geometry = land_use.geometry.buffer(0)


In [ ]:
land_use = land_use.to_crs(tess.crs)


Now we join the land-use data to enclosed tessellation based on the largest intersection.

In [ ]:
%time tess = area_max(land_use, tess, 'use')


We fill any unjoined tessellation cells with `OTHER`.

In [ ]:
tess.use = tess.use.fillna("OTHER")


### 2.4 Parks

Parks are retrieved as OSM `leisure=park` features (points or polygons).

In [ ]:
parks = ox.geometries_from_place(place, tags={'leisure': 'park'}).to_crs(tess.crs)


We measure the distance from each cell to the nearest urban green area (park), same expanding-buffer approach as the Singapore notebook.

In [ ]:
initial_buffer = 500
buffered = tess.buffer(initial_buffer)
distance = []
for orig, geom in tqdm(zip(tess.tessellation, buffered.geometry), total=len(buffered)):
    query = parks.sindex.query(geom, predicate='intersects')
    b = initial_buffer
    while query.size == 0:
        query = parks.sindex.query(geom.buffer(b), predicate='intersects')
        b += initial_buffer

    distance.append(parks.iloc[query].distance(orig).min())


In [ ]:
tess["park_distance"] = distance


### 2.5 Historical monuments

Historical monuments are retrieved via OSM's `historic` tag (monuments, memorials, castles, etc.), linked to the pandana network exactly as eating establishments and supermarkets above.

In [ ]:
%%time
monument_tags = {'historic': ['monument', 'memorial', 'castle', 'fort', 'archaeological_site', 'heritage']}
monuments = ox.geometries_from_place(place, monument_tags).to_crs(tess.crs)
monuments['geometry'] = monuments.geometry.centroid


In [ ]:
network.set_pois(category='monuments',
                 maxdist=1200,
                 maxitems=monuments.shape[0],
                 x_col=monuments.geometry.x,
                 y_col=monuments.geometry.y)


In [ ]:
%%time
results = network.nearest_pois(distance=1200,
                               category='monuments',
                               num_pois=monuments.shape[0],
                               include_poi_ids=False).replace(1200, pd.NA).count(axis=1)


In [ ]:
nodes['monuments'] = results


In [ ]:
nodeid = pd.read_parquet(folder + "tess.pq", columns=["nodeID"])


Finally, we link data attached to nodes to tessellation based on nodeID.

In [ ]:
tess['nodeID'] = nodeid.nodeID.values


In [ ]:
tess = tess.merge(nodes[['eating', 'supermarkets', 'monuments']], on='nodeID', how='left')


In [ ]:
tess.drop(columns=["tessellation", 'nodeID']).to_parquet(folder + "func_data_tess.pq")


### 2.6 Population

We use WorldPop gridded population estimates, same product as the Singapore notebook, just the Switzerland (`CHE`) file instead of Singapore's (`SGP`): https://www.worldpop.org/geodata/summary?id=49730

In [ ]:
path = folder + "che_ppp_2020.tif"
if not os.path.exists(path):
    download("https://data.worldpop.org/GIS/Population/Global_2000_2020/2020/CHE/che_ppp_2020.tif", path)

We open and reproject the file.

In [ ]:
pop = rioxarray.open_rasterio(folder + 'che_ppp_2020.tif')


In [ ]:
pop_proj = pop.rio.reproject(tess.crs)


Now we get an array and affine transformation which we pass to zonal statistics.

In [ ]:
arr = pop_proj.values
affine = pop_proj.rio.transform()


In [ ]:
%%time
stats = rasterstats.zonal_stats(
    tess.geometry,
    raster=arr[0],
    affine=affine,
    stats=['mean'],
    all_touched=True,
)


Then we link mean values to tessellation data.

In [ ]:
tess['population'] = [x['mean'] for x in stats]


Some cells may have a value smaller than 0 due to missingness; we replace those, and any NaN, with 0.

In [ ]:
mask = tess.population < 0
tess.population[mask] = 0


In [ ]:
tess.population = tess.population.fillna(0)


### 2.7 NDVI &mdash; TODO

The Singapore notebook computes NDVI from a Sentinel-2 composite mosaic, using a local tile index (`../../data_processing/imagery/GHS-composite-S2.geojson`) that isn't part of this repository, so the tiling logic can't be carried over as-is.

To implement this for Zurich, the most direct modern replacement would be to query Sentinel-2 L2A scenes for the canton's bounding box from a public STAC catalog (e.g. Microsoft Planetary Computer or Element84's Earth Search), composite/clip to `tess.total_bounds`, compute `(NIR - Red) / (NIR + Red)`, then run the same `rasterstats.zonal_stats(..., stats=['mean'])` pattern used for population above. That requires two extra packages (`pystac-client`, `planetary-computer`) not currently installed, since this step is left unimplemented for now.

In [ ]:
# TODO: implement NDVI for the Canton of Zürich.
# Suggested approach: fetch Sentinel-2 L2A imagery for tess.total_bounds from a public
# STAC catalog (e.g. Microsoft Planetary Computer), compute NDVI = (B08 - B04) / (B08 + B04),
# then zonal_stats(tess.geometry, ndvi_array, affine=..., stats=['mean']) as done for population.
# tess['ndvi'] = ...


### 2.8. Night lights &mdash; TODO

The Singapore notebook's night-lights step requires a manual, login-gated download from https://eogdata.mines.edu/download_dnb_composites.html (VIIRS DNB monthly composite), and even there the tile ID (`75N060E`) is specific to Singapore's location. The equivalent Zurich tile would need to be identified and downloaded by hand from the same portal (an EOG/NASA Earthdata login is required; this cannot be automated from an unattended notebook cell), then processed with the same `rio.clip_box` / `rio.reproject` / `zonal_stats` pattern used in the Singapore notebook.

In [ ]:
# TODO: implement night lights for the Canton of Zürich.
# 1. Manually download the VIIRS DNB monthly composite tile covering Zurich from
#    https://eogdata.mines.edu/download_dnb_composites.html (requires login).
# 2. nl = rioxarray.open_rasterio(folder + '<downloaded_tile>.tif')
# 3. Clip to tess.total_bounds, reproject to tess.crs, and run rasterstats.zonal_stats
#    exactly as done for population/NDVI above.
# tess['lights'] = ...


In [ ]:
tess.drop(columns=["tessellation", 'nodeID']).to_parquet(folder + "func_data_tess.pq")


## 3. Cluster analysis

Cluster analysis uses the data on form, function, and their combination in a K-Means clustering to identify homogenous patterns of built form.

### 3.1. Form

Clustering based on form uses contextualised characters as input.

In [ ]:
form = pd.read_parquet(folder + "convolutions.pq")
tess = gpd.read_parquet(folder + "tess.pq", columns=['tessellation', 'buildings', 'tID'])


In [ ]:
form = form.replace(np.inf, np.nan)  # e.g. due to tiny/edge tessellation cells


Raw data needs to be standardised. We use a standard scaler.

In [ ]:
scaler = preprocessing.StandardScaler()
data = scaler.fit_transform(form)


Any remaining missing values (e.g. at the edges of the study area) are replaced with 0.

In [ ]:
data[np.isnan(data)] = 0


We use clustergram to determine the optimal number of clusters, using 10 initialisations within a range from 1 to 30 clusters.

In [ ]:
cg_form = Clustergram(range(1, 30), backend='sklearn', n_init=10)
cg_form.fit(data)


Now we can plot the resulting clustergram. Due to outliers, we truncate the y axis to see cluster behaviour better &mdash; adjust the limits once you've inspected the actual range for Zurich's data.

In [ ]:
ax = cg_form.plot(figsize=(20, 20), linewidth=0.5, cluster_style={"edgecolor": "r", "alpha": .6}, size=1,
       line_style={"alpha": .5})


Inspect the clustergram and pick the number of clusters where major branching settles down (10 for Singapore; expect a different number here). Before proceeding, we save the clustergram object to a file.

In [ ]:
with open(folder + 'clustergram_form.pickle', 'wb') as f:
    pickle.dump(cg_form, f)


Set `N_FORM_CLUSTERS` to the number you picked from the plot above, then see how the clusters look spatially.

In [ ]:
N_FORM_CLUSTERS = 10  # TODO: adjust based on the clustergram above
tess['clusters_form'] = cg_form.labels[N_FORM_CLUSTERS].values


In [ ]:
ax = tess.plot('clusters_form', categorical=True, legend=True, figsize=(20, 20), cmap='tab20')
ax.set_axis_off()


In [ ]:
ax = tess.set_geometry('buildings').plot('clusters_form', categorical=True, legend=True, figsize=(20, 20), cmap='tab20')
ax.set_axis_off()


### 3.2. Function

Clustering based on function uses only data on function, which we need to prepare.

In [ ]:
function_data = pd.read_parquet(folder + "func_data_tess.pq")


`use` is categorical, so it needs to be encoded as boolean dummies.

In [ ]:
use = pd.get_dummies(function_data.use)


Continuous values need to be standardised.

In [ ]:
scaler = preprocessing.StandardScaler()
f_data = scaler.fit_transform(function_data.drop(columns=["use"]))


In [ ]:
f_data.shape


We merge everything back together.

In [ ]:
f_data = np.hstack([f_data, use.values])


In [ ]:
f_data.shape


In [ ]:
np.isnan(f_data).any().any()


We use clustergram again.

In [ ]:
cg_function = Clustergram(range(1, 30), backend='sklearn', n_init=10)
cg_function.fit(f_data)


In [ ]:
ax = cg_function.plot(figsize=(20, 20), linewidth=0.5, cluster_style={"edgecolor": "r", "alpha": .6}, size=1,
       line_style={"alpha": .5})


Clustergram should be saved to a file.

In [ ]:
with open(folder + 'clustergram_function.pickle', 'wb') as f:
    pickle.dump(cg_function, f)


Again, pick the optimal number of clusters from the plot above.

In [ ]:
N_FUNCTION_CLUSTERS = 9  # TODO: adjust based on the clustergram above
tess['clusters_function'] = cg_function.labels[N_FUNCTION_CLUSTERS].values


In [ ]:
ax = tess.plot('clusters_function', categorical=True, legend=True, figsize=(20, 20), cmap='tab20')
ax.set_axis_off()


In [ ]:
ax = tess.set_geometry('buildings').plot('clusters_function', categorical=True, legend=True, figsize=(20, 20), cmap='tab20')
ax.set_axis_off()


### 3.3. Spatial Signatures

Final clustering combines both form and function into a single classification. First we stack both preprocessed arrays together and then run clustergram.

In [ ]:
ss_data = np.hstack([data, f_data])


In [ ]:
cg_ss = Clustergram(range(1, 30), backend='sklearn', n_init=10, random_state=0)
cg_ss.fit(ss_data)


In [ ]:
c = ugg.COLORS
ax = cg_ss.plot(
    figsize=(12, 12),
    linewidth=0.5,
    cluster_style={"edgecolor": "w", "color": c[2], "linewidth": 1.5},
    size=1,
    line_style={"color": c[1]},
    pca_kwargs=dict(random_state=0),
)
plt.savefig(folder + "cgram.png", dpi=300, bbox_inches="tight")


In [ ]:
cg_labels = cg_ss.labels
cg_labels.columns = cg_labels.columns.astype(str)
cg_labels.to_parquet(folder + 'clustergram_ss_labels.pq')


Pick the optimal number of clusters from the plot above (Singapore settled on 12), then run the final clustering with many initialisations and a fixed random state for reproducibility.

In [ ]:
N_SS_CLUSTERS = 12  # TODO: adjust based on the clustergram above
%time km = MiniBatchKMeans(n_clusters=N_SS_CLUSTERS, n_init=1000, random_state=12345, batch_size=100000).fit(ss_data)

In [ ]:
tess['clusters'] = km.labels_


In [ ]:
ax = tess.plot('clusters', categorical=True, legend=True, figsize=(20, 20), cmap='tab20')
ax.set_axis_off()


In [ ]:
ax = tess.set_geometry('buildings').plot('clusters', categorical=True, legend=True, figsize=(20, 20), cmap='tab20')
ax.set_axis_off()


In [ ]:
tess[['tID', 'clusters', 'clusters_form', 'clusters_function']].to_parquet(folder + "cluster_labels.pq")
tess[['tID', 'clusters']].to_csv(folder + "FINAL_cluster_labels.csv")


#### Generate signatures

As a final step, we generate a geometry representing spatial signatures as a combination of contiguous tessellation cells belonging to a single cluster. Since we have to dissolve a large number of polygons, we use `dask_geopandas` to parallelise the procedure. First we create a local cluster.

In [ ]:
client = Client(LocalCluster(n_workers=8))
client


We read only the `tessellation` column and cluster labels. The resulting geometry will no longer be enclosed tessellation, so we rename it to a generic `geometry` already.

In [ ]:
tess = gpd.read_parquet(folder + "tess.pq", columns=['tessellation']).rename_geometry("geometry")
clusters = pd.read_csv(folder + "FINAL_cluster_labels.csv", index_col=0)


In [ ]:
tess['cluster'] = clusters.clusters.values


`dask_dissolve` mimics the behaviour of `geopandas.dissolve`, based on a parallel implementation using `dask-geopandas`.

In [ ]:
%%time
ddf = dask_geopandas.from_geopandas(tess.sort_values('cluster'), npartitions=64)
final = dask_dissolve(ddf, by='cluster').compute()


In [ ]:
final.plot('cluster', categorical=True, figsize=(20, 20), cmap='tab20')


Finally, we explode the geometries to get multiple signatures of the same kind instead of a single multipolygon. (The initial `reset_index` avoids a geopandas bug.)

In [ ]:
final.reset_index(drop=True).explode(index_parts=False).reset_index(drop=True).to_parquet(folder + "signatures.pq")


Before we end, we close the dask client.

In [ ]:
client.close()


# 4. Figures

Finally, we create the final figures. Singapore's notebook hardcodes basemap tile bounds (`ax.set_xlim(11530000, 11590000)`) specific to Singapore's extent; here we derive the bounds from the data instead so this works for any canton shape.

Note on basemaps: the original notebook uses `urbangrammar_graphics.get_tiles(...)`, which requires a personal Mapbox access token (the `token = ""` placeholder below is non-functional as-is, same as in the Singapore notebook). If you don't have a Mapbox token, swap `contextily.add_basemap(..., source=ugg.get_tiles(...))` for a free provider, e.g. `contextily.add_basemap(ax, source=contextily.providers.CartoDB.Positron)`.

In [ ]:
final = gpd.read_parquet(folder + "signatures.pq")


In [ ]:
enc = gpd.read_parquet(folder + "enclosures.pq")


In [ ]:
sns.set(context="paper", style="ticks", rc={'patch.force_edgecolor': False})


In [ ]:
cmap = ugg.get_colormap(final.cluster.nunique() + 1, randomize=False)


In [ ]:
gradient = np.linspace(0, 1, 256)
gradient = np.vstack((gradient, gradient))

fig, ax = plt.subplots(figsize=(8, .5))
ax.imshow(gradient, aspect='auto', cmap=cmap)


Symbology mapping cluster id to a colour. Adjust the number of entries to match `N_SS_CLUSTERS` chosen above.

In [ ]:
cols = cmap.colors
symbology = {i: cols[i] for i in range(final.cluster.nunique())}


In [ ]:
df = final.set_crs(enc.crs).to_crs(3857)


In [ ]:
token = ""  # requires a personal Mapbox access token; see note above for a token-free alternative


In [ ]:
minx, miny, maxx, maxy = df.total_bounds

ax = df.plot(color=df['cluster'].map(symbology), figsize=(20, 20), zorder=1, linewidth=.3, edgecolor='w', alpha=1)
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
if token:
    contextily.add_basemap(ax, crs=df.crs, source=ugg.get_tiles('roads', token), zorder=2, alpha=.3)
    contextily.add_basemap(ax, crs=df.crs, source=ugg.get_tiles('labels', token), zorder=3, alpha=1)
    contextily.add_basemap(ax, crs=df.crs, source=ugg.get_tiles('background', token), zorder=-1, alpha=1)
else:
    contextily.add_basemap(ax, crs=df.crs, source=contextily.providers.CartoDB.Positron, zorder=-1, alpha=1)
ax.set_axis_off()

scalebar = ScaleBar(dx=1,
                    color=ugg.COLORS[0],
                    location='lower right',
                    height_fraction=0.002,
                    pad=.5,
                    frameon=False,
                    )
ax.add_artist(scalebar)

ugg.north_arrow(plt.gcf(), ax, 0, size=.026, linewidth=1, color=ugg.COLORS[0], loc="upper left", pad=.002, alpha=.9)

from matplotlib.lines import Line2D
custom_points = [Line2D([0], [0], marker="o", linestyle="none", markersize=10, color=color) for color in symbology.values()]
leg_points = ax.legend(custom_points, symbology.keys(), loc='upper right', frameon=True)
ax.add_artist(leg_points)
plt.savefig(folder + "signatures.png", dpi=300, bbox_inches="tight")


In [ ]:
tess = gpd.read_parquet(folder + "tess.pq").set_geometry('buildings')
clusters = pd.read_csv(folder + "FINAL_cluster_labels.csv", index_col=0)
tess['cluster'] = clusters.clusters.values


In [ ]:
tess_wm = tess.to_crs(3857)
minx, miny, maxx, maxy = tess_wm.total_bounds

ax = tess_wm.plot(color=tess_wm['cluster'].map(symbology), figsize=(20, 20), zorder=1, alpha=1)
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
if token:
    contextily.add_basemap(ax, crs=df.crs, source=ugg.get_tiles('roads', token), zorder=2, alpha=.3)
    contextily.add_basemap(ax, source=ugg.get_tiles('labels', token), zorder=3, alpha=1)
    contextily.add_basemap(ax, source=ugg.get_tiles('background', token), zorder=-1, alpha=1)
else:
    contextily.add_basemap(ax, source=contextily.providers.CartoDB.Positron, zorder=-1, alpha=1)
ax.set_axis_off()

scalebar = ScaleBar(dx=1,
                    color=ugg.COLORS[0],
                    location='lower right',
                    height_fraction=0.002,
                    pad=.5,
                    frameon=False,
                    )
ax.add_artist(scalebar)

ugg.north_arrow(plt.gcf(), ax, 0, size=.026, linewidth=1, color=ugg.COLORS[0], loc="upper left", pad=.002, alpha=.9)

from matplotlib.lines import Line2D
custom_points = [Line2D([0], [0], marker="o", linestyle="none", markersize=10, color=color) for color in symbology.values()]
leg_points = ax.legend(custom_points, symbology.keys(), loc='upper right', frameon=True)
ax.add_artist(leg_points)
plt.savefig(folder + "signatures_buildings.png", dpi=300, bbox_inches="tight")


In [ ]:
clusters = pd.read_csv(folder + "FINAL_cluster_labels.csv", index_col=0)


In [ ]:
counts = clusters.clusters.value_counts(normalize=True)
color = counts.index.map(symbology)
counts.plot.bar(color=color)
plt.ylabel("ratio")
plt.savefig(folder + "counts.pdf", dpi=300, bbox_inches="tight")


## 5. Cluster summary

We can also export summary values of our input characters for each cluster.

In [ ]:
data = gpd.read_parquet(folder + "data.pq")


We list the characters we want to summarise.

In [ ]:
characters = [
    "stcOri",
    "stbOri",
    "stbCeA",
    "sdcLAL",
    "sdcAre",
    "sscCCo",
    "sscERI",
    "sicCAR",
    "mtbAli",
    "mtbNDi",
    "mtcWNe",
    "mdcAre",
    "ltbIBD",
    "stcSAl",
    "ltcWRE",
    "sdbAre",
    "sdbPer",
    "sdbCoA",
    "ssbCCo",
    "ssbCor",
    "ssbSqu",
    "ssbERI",
    "ssbElo",
    "ssbCCM",
    "ssbCCD",
    "mtbSWR",
    "ldbPWL",
    "stbSAl",
    "libNCo",
    "sdsLen",
    "sdsSPW",
    "sdsSPO",
    "sdsSWD",
    "sssLin",
    "sdsAre",
    "sisBpM",
    "misRea",
    "mdsAre",
    "ldsMSL",
    "degree",
    "meshedness",
    "proportion_3",
    "proportion_4",
    "proportion_0",
    "local_closeness",
    "ldsCDL",
    "xcnSCl",
    "mtdMDi",
    "lddNDe",
    "linWID",
    "sddAre",
    "ldeAre",
    "ldePer",
    "lseCCo",
    "lseERI",
    "lseCWA",
    "lteOri",
    "lteWNB",
    "lieWCe",
]


In [ ]:
clusters = pd.read_csv(folder + "FINAL_cluster_labels.csv", index_col=0)


In [ ]:
data["cluster"] = clusters.clusters.values


We get the mean of each character for each cluster.

In [ ]:
means = data[characters].groupby(data.cluster).mean().T


It is better to use full names, as the result should be interpretable. Codes are not that useful on their own.

In [ ]:
char_names = {
    'sdbAre': 'area of building',
    'sdbPer': 'perimeter of building',
    'sdbCoA': 'courtyard area of building',
    'ssbCCo': 'circular compactness of building',
    'ssbCor': 'corners of building',
    'ssbSqu': 'squareness of building',
    'ssbERI': 'equivalent rectangular index of building',
    'ssbElo': 'elongation of building',
    'ssbCCM': 'centroid - corner mean distance of building',
    'ssbCCD': 'centroid - corner distance deviation of building',
    'stbOri': 'orientation of building',
    'sdcLAL': 'longest axis length of ETC',
    'sdcAre': 'area of ETC',
    'sscCCo': 'circular compactness of ETC',
    'sscERI': 'equivalent rectangular index of ETC',
    'stcOri': 'orientation of ETC',
    'sicCAR': 'covered area ratio of ETC',
    'stbCeA': 'cell alignment of building',
    'mtbAli': 'alignment of neighbouring buildings',
    'mtbNDi': 'mean distance between neighbouring buildings',
    'mtcWNe': 'perimeter-weighted neighbours of ETC',
    'mdcAre': 'area covered by neighbouring cells',
    'ltcWRE': 'weighted reached enclosures of ETC',
    'ltbIBD': 'mean inter-building distance',
    'sdsSPW': 'width of street profile',
    'sdsSWD': 'width deviation of street profile',
    'sdsSPO': 'openness of street profile',
    'sdsLen': 'length of street segment',
    'sssLin': 'linearity of street segment',
    'ldsMSL': 'mean segment length within 3 steps',
    'degree': 'node degree of junction',
    'meshedness': 'local meshedness of street network',
    'proportion_3': 'local proportion of 3-way intersections of street network',
    'proportion_4': 'local proportion of 4-way intersections of street network',
    'proportion_0': 'local proportion of cul-de-sacs of street network',
    'local_closeness': 'local closeness of street network',
    'ldsCDL': 'local cul-de-sac length of street network',
    'xcnSCl': 'square clustering of street network',
    'mtdMDi': 'mean distance to neighbouring nodes of street network',
    'lddNDe': 'local node density of street network',
    'linWID': 'local degree weighted node density of street network',
    'stbSAl': 'street alignment of building',
    'stcSAl': 'street alignment of ETC',
    'mtbSWR': 'shared walls ratio of buildings',
    'sddAre': 'area covered by node-attached ETCs',
    'sdsAre': 'area covered by edge-attached ETCs',
    'sisBpM': 'buildings per meter of street segment',
    'mdsAre': 'reached area by neighbouring segments',
    'ldeAre': 'area of enclosure',
    'ldePer': 'perimeter of enclosure',
    'lseCCo': 'circular compactness of enclosure',
    'lseERI': 'equivalent rectangular index of enclosure',
    'lseCWA': 'compactness-weighted axis of enclosure',
    'lteOri': 'orientation of enclosure',
    'lteWNB': 'perimeter-weighted neighbours of enclosure',
    'lieWCe': 'area-weighted ETCs of enclosure',
    'ldbPWL': 'perimeter wall length of adjacent buildings',
    'libNCo': 'number of courtyards within adjacent buildings',
    'misRea': 'cells reached within neighbouring street segments',
}


In [ ]:
means.index = means.index.map(char_names)


We do the same with function characters.

In [ ]:
function_data = pd.read_parquet(folder + "func_data_tess.pq")


Function columns for Zurich: `use` (categorical, excluded here), `park_distance`, `eating`, `supermarkets`, `monuments`, `population` (NDVI/lights are TODO, see Section 2.7-2.8, and are absent until implemented).

In [ ]:
f_cols = [c for c in ["park_distance", "eating", "supermarkets", "monuments", "population", "ndvi", "lights"] if c in function_data.columns]
f_means = function_data[f_cols].groupby(data.cluster.values).mean().T


In [ ]:
rename = {
    "park_distance": "distance to nearest park",
    "eating": "eating establishments",
    "supermarkets": "supermarkets",
    "monuments": "historical monuments",
    "population": "population",
    "ndvi": "NDVI",
    "lights": "night lights",
}
f_means.index = [rename[c] for c in f_cols]


Finally, we join form and function summaries and plot the final output as a styled pandas DataFrame.

In [ ]:
together = pd.concat([means, f_means])


In [ ]:
together.replace(np.inf, np.nan).fillna(0).style.format('{:.2f}').background_gradient(cmap=sns.diverging_palette(235, 16, 60, 55, as_cmap=True), axis=1)


## Create a data product

Create files that could be uploaded to a public repository (e.g. figshare) for public access: the raw measured data at the enclosed-tessellation-cell level, and the signature geometry.

In [ ]:
form = gpd.read_parquet(folder + "data.pq")
function = pd.read_parquet(folder + "func_data_tess.pq")
conv = pd.read_parquet(folder + "convolutions.pq")


In [ ]:
function


In [ ]:
combined = pd.concat([form, conv, function.drop(columns=["tID"])], axis=1)


In [ ]:
clusters = pd.read_csv(folder + "FINAL_cluster_labels.csv", index_col=0)


In [ ]:
combined["signature_type"] = clusters.clusters.values


In [ ]:
combined.to_parquet(folder + "zurich_data.parquet")


In [ ]:
sig = gpd.read_parquet(folder + "signatures.pq")


In [ ]:
sig.crs = combined.crs


In [ ]:
sig = sig.rename(columns={"cluster": "signature_type"})


In [ ]:
sig.to_parquet(folder + "signatures.pq")
